## Introduction to deep learning - exercise

## Example

*Human Activity Recognition using Smartphones* dataset

Dataset description:

*The experiments have been carried out with a group of 30 volunteers. Each person performed six activities
(WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING) wearing a smartphone.
Using its embedded accelerometer and gyroscope, we captured 3-axial linear acceleration and 3-axial angular velocity.
The experiments have been video-recorded to label the data manually.*

**Variables:**
For each record in the dataset it is provided:
* A 561-feature vector with time and frequency domain variables.
* Its activity label.
* An identifier of the subject who carried out the experiment.

More details at: https://archive.ics.uci.edu/ml/datasets/human+activity+recognition+using+smartphones

### Loading and preparing the data

In [2]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim

In [1]:
folder = '.'  ## put here folder where the file HAR_clean.csv is located

Load the dataset: "HAR_clean.csv"

In [3]:
all_data = pd.read_csv(os.path.join(folder, 'HAR_clean.csv'), index_col=0)

Divide into input and output data

In [4]:
input_data = all_data.iloc[:,:-2].values.astype(np.float32)
input_data.shape

(10299, 561)

In [5]:
output_data = all_data.iloc[:,-1].values
output_data.shape

(10299,)

Divide the data into train and test, keeping 30% for the test

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split the data into training and testing sets
##########################################

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    input_data, output_data, test_size=0.2, random_state=42
)

###########################################

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Best shallow ML model

In [10]:
from sklearn import svm
from sklearn.model_selection import GridSearchCV

parameters = {'kernel':['linear', 'rbf'], 'C':[1, 10, 100,1000], 'gamma':[0.01, 0.001]}

svm_model_d = svm.SVC()
opt_model_d = GridSearchCV(svm_model_d, parameters)
print('boas')
opt_model_d.fit(X_train, y_train)
print('boas')
print (opt_model_d.best_estimator_)

boas


KeyboardInterrupt: 

In [ ]:
opt_model_d.score(X_test, y_test)

0.983495145631068

### Training a DL model

In [11]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()
# LabelEncoder() is needed for DL models, as they require numerical labels
# sklearn does that internally for SVM, but we need to do it explicitly for DL models

le.fit(y_train)

y_train_encoded = le.transform(y_train)
y_test_encoded = le.transform(y_test)

In [12]:
X_train = torch.tensor(X_train)
y_train = torch.tensor(y_train_encoded)

X_test = torch.tensor(X_test)
y_test = torch.tensor(y_test_encoded)

In [27]:
# DNN model - define here the model - complete

class HARNet(nn.Module):
    def __init__(self, in_features, n_classes):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,n_classes),
            nn.Dropout(0.2)
        )

    def forward(self, x):
        return self.net(x)


n_features = X_train.shape[1]
n_classes = len(np.unique(y_train))

model = HARNet(n_features, n_classes)

In [34]:
# Loss & optimizer - complete !
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)

In [38]:
from sklearn.metrics import accuracy_score

# Training loop
epochs = 23
batch_size = 64
train_losses = []
test_accuracies = []

for epoch in range(epochs):
    model.train()
    # ---- mini-batch training ----
    perm = torch.randperm(X_train.size(0))
    epoch_loss = 0

    for i in range(0, X_train.size(0), batch_size):
        idx = perm[i:i + batch_size]
        xb, yb = X_train[idx], y_train[idx]

        preds = model(xb)
        loss = criterion(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    train_losses.append(epoch_loss)

    # ---- evaluation ----
    model.eval()
    with torch.no_grad():
        logits = model(X_test)
        preds = torch.argmax(logits, dim=1)
        acc = accuracy_score(y_test.numpy(), preds.numpy())
        test_accuracies.append(acc)

    print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss:.3f} | Test Acc: {acc:.4f}")

Epoch 01 | Loss: 15.626 | Test Acc: 0.9864
Epoch 02 | Loss: 15.235 | Test Acc: 0.9869
Epoch 03 | Loss: 15.732 | Test Acc: 0.9874
Epoch 04 | Loss: 15.496 | Test Acc: 0.9874
Epoch 05 | Loss: 14.924 | Test Acc: 0.9883
Epoch 06 | Loss: 16.006 | Test Acc: 0.9874
Epoch 07 | Loss: 15.108 | Test Acc: 0.9854
Epoch 08 | Loss: 16.115 | Test Acc: 0.9874
Epoch 09 | Loss: 16.181 | Test Acc: 0.9879
Epoch 10 | Loss: 15.829 | Test Acc: 0.9879
Epoch 11 | Loss: 15.096 | Test Acc: 0.9874
Epoch 12 | Loss: 16.417 | Test Acc: 0.9874
Epoch 13 | Loss: 15.135 | Test Acc: 0.9883
Epoch 14 | Loss: 15.534 | Test Acc: 0.9879
Epoch 15 | Loss: 15.549 | Test Acc: 0.9869
Epoch 16 | Loss: 16.029 | Test Acc: 0.9879
Epoch 17 | Loss: 15.423 | Test Acc: 0.9874
Epoch 18 | Loss: 16.162 | Test Acc: 0.9879
Epoch 19 | Loss: 15.509 | Test Acc: 0.9883
Epoch 20 | Loss: 15.323 | Test Acc: 0.9883
Epoch 21 | Loss: 15.572 | Test Acc: 0.9888
Epoch 22 | Loss: 15.162 | Test Acc: 0.9879
Epoch 23 | Loss: 15.401 | Test Acc: 0.9879


In [39]:
# Test set evaluation
print("\nFinal test accuracy:", test_accuracies[-1])


Final test accuracy: 0.9878640776699029
